# Temperatura ambiental efectiva: versió final i presentable

Objectiu: construir una coordenada ambiental efectiva que ordeni millor els vots estàtics de **sensació tèrmica** i **comfort** que la temperatura corregida sola.

La idea és seqüencial:

\[
T_{rad}=T_{corr}+k_{mix}f_{mix}+k_{sun}f_{sun}
\]

\[
T_{env}^{*}=T_{rad}-k_{wind}wind_{sc}
\]

**Important:** \(T_{env}^{*}\) no s'interpreta com una temperatura física exacta en graus, sinó com una **coordenada ambiental efectiva** calibrada empíricament. Si algun coeficient arriba al límit del grid, la seva magnitud no s'ha de llegir literalment com un offset físic.

In [ ]:
# ============================================================
# 0. Imports i configuració
# ============================================================
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import kruskal
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# ----------------------- CONFIG -----------------------
# Canvia només això si cal.
CSV_CANDIDATES = [
    Path('../../data/all_surveys(stops).csv'),
    Path('data/all_surveys(stops)(2).csv'),
    Path('/mnt/data/all_surveys(stops).csv'),
    Path('/mnt/data/all_surveys(stops)(2).csv'),
]

OUTDIR = Path('effective_temperature_results_final')
OUTDIR.mkdir(exist_ok=True)

# Grid finit: no busquem una "temperatura real", sinó una coordenada efectiva interpretable.
K_MAX = 20.0
K_STEP = 0.5
K_GRID = np.arange(0, K_MAX + 1e-9, K_STEP)

# Per validació creuada, un pas més gruixut fa que corri molt més ràpid.
RUN_CV = True
K_GRID_CV = np.arange(0, K_MAX + 1e-9, 1.0)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# ============================================================
# 1. Definicions de vots i variables ambientals
# ============================================================
COMFORT_SCORE = {
    'Very uncomfortable': -3,
    'Uncomfortable': -2,
    'Slightly uncomfortable': -1,
    'Neutral(comfort)': 0,
    'Slightly comfortable': 1,
    'Comfortable': 2,
    'Very comfortable': 3,
}
COMFORT3 = {
    'Very uncomfortable': 'uncomfortable',
    'Uncomfortable': 'uncomfortable',
    'Slightly uncomfortable': 'uncomfortable',
    'Neutral(comfort)': 'neutral',
    'Slightly comfortable': 'comfortable',
    'Comfortable': 'comfortable',
    'Very comfortable': 'comfortable',
}

SENS_SCORE = {
    'Cool': -2,
    'Slightly cool': -1,
    'Neutral(sensation)': 0,
    'Slightly warm': 1,
    'Warm': 2,
    'Hot': 3,
    'Very hot': 4,
}
SENS3 = {
    'Cool': 'cool',
    'Slightly cool': 'cool',
    'Neutral(sensation)': 'neutral',
    'Slightly warm': 'hot',
    'Warm': 'hot',
    'Hot': 'hot',
    'Very hot': 'hot',
}

SUN = ['In full sun', 'In a mixture of sun and shadow', 'In full shade']
WIND = ["It's not windy", 'A very light wind', 'A light wind', 'A moderate wind', 'A strong wind']

T_COL   = '<T-T_fixed+<T>>'
HDX_COL = '<HDX-HDX_fixed+<HDX>>'

In [ ]:
# ============================================================
# 2. Càrrega i neteja mínima
# ============================================================
def find_csv(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('No he trobat el CSV. Canvia CSV_CANDIDATES a la cel·la de configuració.')

CSV = find_csv(CSV_CANDIDATES)
print('Llegint:', CSV)

df = pd.read_csv(CSV)

# Normalització mínima.
df['city'] = df['city'].astype(str).str.replace('Llobreat', 'Llobregat', regex=False).str.strip()
df['walk'] = df['date'].astype(str) + '|' + df['driver_name'].astype(str) + '|' + df['city'].astype(str)

# Variables ambientals base.
df['T_corr'] = pd.to_numeric(df[T_COL], errors='coerce')
df['HDX_corr'] = pd.to_numeric(df[HDX_COL], errors='coerce')
df['SVF_n'] = pd.to_numeric(df.get('SVF', np.nan), errors='coerce')

# Sol: fraccions de vots de cada condició solar a la parada.
df[SUN] = df[SUN].fillna(0)
sun_tot = df[SUN].sum(axis=1).replace(0, np.nan)
df['f_sun'] = df['In full sun'] / sun_tot
df['f_mix'] = df['In a mixture of sun and shadow'] / sun_tot

# Vent: índex ordinal 0..1.
df[WIND] = df[WIND].fillna(0)
wind_tot = df[WIND].sum(axis=1)
wind_weighted = (df[WIND] * np.array([0, 1, 2, 3, 4])).sum(axis=1)
df['wind_sc'] = np.where(wind_tot > 0, wind_weighted / (wind_tot * 4), 0.0)

# Neteja final.
df = df.dropna(subset=['T_corr', 'HDX_corr', 'f_sun', 'f_mix', 'wind_sc']).reset_index(drop=True)

print(f'parades: {len(df)} | caminades: {df.walk.nunique()} | ciutats: {df.city.nunique()}')
print(f'T_corr: {df.T_corr.min():.1f} - {df.T_corr.max():.1f} ºC')
print(f'f_sun:  {df.f_sun.min():.2f} - {df.f_sun.max():.2f}')
print(f'f_mix:  {df.f_mix.min():.2f} - {df.f_mix.max():.2f}')
print(f'wind:   {df.wind_sc.min():.2f} - {df.wind_sc.max():.2f}')

## Què calibrem?

Treballarem a nivell de **vot individual**: cada vot hereta les covariables ambientals de la seva parada.

Es fan dues anàlisis separades:

- **Sensació tèrmica**: outcome principal. Més \(T_{env}^{*}\) hauria d'implicar vots més calents.
- **Comfort**: outcome secundari. En règim calorós, més \(T_{env}^{*}\) hauria d'implicar menys comfort.

In [ ]:
# ============================================================
# 3. Expansió a nivell de vot
# ============================================================
def expand_votes(df, score_map, state3_map, outcome_name):
    rows = []
    vote_cols = [c for c in score_map if c in df.columns]
    missing = [c for c in score_map if c not in df.columns]
    if missing:
        print(f'ATENCIÓ {outcome_name}: columnes no trobades:', missing)

    for idx, r in df.iterrows():
        for c in vote_cols:
            n = int(round(0 if pd.isna(r[c]) else r[c]))
            if n <= 0:
                continue
            rows.extend([{
                'outcome': outcome_name,
                'stop_row': idx,
                'walk': r['walk'],
                'city': r['city'],
                'space_name': r.get('space_name', ''),
                'category': c,
                'score': score_map[c],
                'state3': state3_map[c],
                'T_corr': r['T_corr'],
                'HDX_corr': r['HDX_corr'],
                'f_mix': r['f_mix'],
                'f_sun': r['f_sun'],
                'wind_sc': r['wind_sc'],
                'SVF_n': r['SVF_n'],
            }] * n)

    v = pd.DataFrame(rows)
    return v

v_sens = expand_votes(df, SENS_SCORE, SENS3, 'sensation')
v_comf = expand_votes(df, COMFORT_SCORE, COMFORT3, 'comfort')

print('vots sensation:', len(v_sens), v_sens['state3'].value_counts().to_dict())
print('vots comfort:  ', len(v_comf), v_comf['state3'].value_counts().to_dict())

In [ ]:
# ============================================================
# 4. Funcions de mètrica i grid search
# ============================================================
def spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or len(np.unique(x[mask])) < 2 or len(np.unique(y[mask])) < 2:
        return np.nan
    return stats.spearmanr(x[mask], y[mask]).statistic


def auc_from_coordinate(x, target):
    x = np.asarray(x, dtype=float)
    target = np.asarray(target, dtype=int)
    mask = np.isfinite(x) & np.isfinite(target)
    if len(np.unique(target[mask])) < 2:
        return np.nan
    return roc_auc_score(target[mask], x[mask])


def kw_state3(x, state3):
    tmp = pd.DataFrame({'x': x, 'state3': state3}).dropna()
    groups = [g['x'].values for _, g in tmp.groupby('state3') if len(g) >= 3]
    if len(groups) < 2:
        return np.nan
    return kruskal(*groups).statistic


def add_binary_target(v, outcome):
    v = v.copy()
    if outcome == 'sensation':
        # En condicions càlides, interessa separar el costat warm/hot del neutral/cool.
        v['binary_target'] = (v['score'] > 0).astype(int)
        target_name = 'P(warm/hot sensation)'
        direction = +1  # més temperatura efectiva -> més sensació calenta
    elif outcome == 'comfort':
        v['binary_target'] = (v['score'] < 0).astype(int)
        target_name = 'P(discomfort)'
        direction = -1  # més temperatura efectiva -> menys comfort
    else:
        raise ValueError(outcome)
    return v, target_name, direction


def objective_rho(x, score, direction):
    # Convertim sempre la mètrica a "més alt és millor".
    # sensation: direction=+1, volem rho positiu.
    # comfort:   direction=-1, volem rho negatiu, per tant maximitzem -rho.
    rho = spearman(x, score)
    return direction * rho


def fit_T_rad(v, direction, k_grid=K_GRID):
    best = {'k_mix': 0.0, 'k_sun': 0.0, 'rho_raw': np.nan, 'objective': -np.inf}
    y = v['score'].values
    T = v['T_corr'].values
    f_mix = v['f_mix'].values
    f_sun = v['f_sun'].values

    for k_mix in k_grid:
        base_mix = T + k_mix * f_mix
        for k_sun in k_grid:
            x = base_mix + k_sun * f_sun
            obj = objective_rho(x, y, direction)
            if np.isfinite(obj) and obj > best['objective']:
                best = {'k_mix': k_mix, 'k_sun': k_sun, 'rho_raw': spearman(x, y), 'objective': obj}
    return best


def fit_T_env(v, direction, k_grid=K_GRID):
    best = {'k_mix': 0.0, 'k_sun': 0.0, 'k_wind': 0.0, 'rho_raw': np.nan, 'objective': -np.inf}
    y = v['score'].values
    T = v['T_corr'].values
    f_mix = v['f_mix'].values
    f_sun = v['f_sun'].values
    wind = v['wind_sc'].values

    for k_mix in k_grid:
        base_mix = T + k_mix * f_mix
        for k_sun in k_grid:
            base_sun = base_mix + k_sun * f_sun
            for k_wind in k_grid:
                x = base_sun - k_wind * wind
                obj = objective_rho(x, y, direction)
                if np.isfinite(obj) and obj > best['objective']:
                    best = {
                        'k_mix': k_mix,
                        'k_sun': k_sun,
                        'k_wind': k_wind,
                        'rho_raw': spearman(x, y),
                        'objective': obj,
                    }
    return best


def apply_models(v, rad_params, env_params):
    v = v.copy()
    v['T_rad'] = (
        v['T_corr']
        + rad_params['k_mix'] * v['f_mix']
        + rad_params['k_sun'] * v['f_sun']
    )
    v['T_env'] = (
        v['T_corr']
        + env_params['k_mix'] * v['f_mix']
        + env_params['k_sun'] * v['f_sun']
        - env_params['k_wind'] * v['wind_sc']
    )
    return v


def metric_table(v, direction):
    rows = []
    for col in ['T_corr', 'HDX_corr', 'T_rad', 'T_env']:
        raw_rho = spearman(v[col], v['score'])
        rows.append({
            'variable': col,
            'rho_raw': raw_rho,
            'objective_rho': direction * raw_rho,
            'AUC_binary': auc_from_coordinate(v[col], v['binary_target']),
            'Kruskal_H_state3': kw_state3(v[col], v['state3']),
        })
    return pd.DataFrame(rows)

## Model final

Fem servir un grid finit \([0,20]\) per evitar que la variable es converteixi en un classificador sense escala. Si el màxim cau a la frontera, ho reportem com una limitació: els coeficients són pesos empírics, no graus físics exactes.

In [ ]:
# ============================================================
# 5. Anàlisi principal: sensation i comfort
# ============================================================
def run_main_analysis(v, outcome):
    v, target_name, direction = add_binary_target(v, outcome)

    print('\n' + '='*70)
    print(outcome.upper())
    print('='*70)
    print('Target AUC:', target_name)
    print('Direcció esperada:', 'rho positiu' if direction == 1 else 'rho negatiu')

    rad = fit_T_rad(v, direction, K_GRID)
    env = fit_T_env(v, direction, K_GRID)
    v2 = apply_models(v, rad, env)
    tab = metric_table(v2, direction)

    print('\nT_rad = T_corr + k_mix f_mix + k_sun f_sun')
    print(f"  k_mix = {rad['k_mix']:.2f} ºC | k_sun = {rad['k_sun']:.2f} ºC | objective rho = {rad['objective']:.3f}")

    print('\nT_env* = T_corr + k_mix f_mix + k_sun f_sun - k_wind wind_sc')
    print(f"  k_mix = {env['k_mix']:.2f} ºC | k_sun = {env['k_sun']:.2f} ºC | k_wind = {env['k_wind']:.2f} ºC | objective rho = {env['objective']:.3f}")

    # Avisos de frontera.
    boundary = []
    for name in ['k_mix', 'k_sun', 'k_wind']:
        if name in env and np.isclose(env[name], K_MAX):
            boundary.append(name)
    if boundary:
        print('\nATENCIÓ: coeficients a la frontera del grid:', boundary)
        print('Interpretació: pesos empírics de separació, NO offsets físics exactes.')

    display(tab)

    tab.to_csv(OUTDIR / f'metrics_{outcome}.csv', index=False)
    v2.to_csv(OUTDIR / f'votes_with_Tenv_{outcome}.csv', index=False)

    coef = pd.DataFrame([
        {'outcome': outcome, 'model': 'T_rad', **rad},
        {'outcome': outcome, 'model': 'T_env', **env},
    ])
    coef.to_csv(OUTDIR / f'grid_coefficients_{outcome}.csv', index=False)

    return v2, tab, coef

v_sens_final, tab_sens, coef_sens = run_main_analysis(v_sens, 'sensation')
v_comf_final, tab_comf, coef_comf = run_main_analysis(v_comf, 'comfort')

summary = pd.concat([
    tab_sens.assign(outcome='sensation'),
    tab_comf.assign(outcome='comfort'),
], ignore_index=True)
summary.to_csv(OUTDIR / 'summary_metrics_all.csv', index=False)
summary

## Figura presentable

Evitem usar com a figura principal una corba binned de probabilitat, perquè pot perdre monotonia per soroll de binning i perquè \(T_{env}^{*}\) reordena els punts. És més clar mostrar si les tres variables separen els estats perceptius.

In [ ]:
# ============================================================
# 6. Figura: separació dels estats state3
# ============================================================
def plot_state_separation(v, outcome, order=None):
    variables = ['T_corr', 'T_rad', 'T_env']
    if order is None:
        order = list(v['state3'].dropna().unique())

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False)
    for ax, var in zip(axes, variables):
        data = [v.loc[v['state3'] == s, var].dropna().values for s in order]
        ax.boxplot(data, labels=order, showfliers=False)
        ax.set_title(var)
        ax.set_xlabel('state3')
        ax.set_ylabel('temperature coordinate (ºC)')
        ax.tick_params(axis='x', rotation=25)

    fig.suptitle(f'{outcome}: separation of perceptual states', y=1.04)
    plt.tight_layout()
    plt.savefig(OUTDIR / f'boxplot_state_separation_{outcome}.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_state_separation(v_sens_final, 'sensation', order=['cool', 'neutral', 'hot'])
plot_state_separation(v_comf_final, 'comfort', order=['uncomfortable', 'neutral', 'comfortable'])

## Validació creuada senzilla per caminada

Aquesta cel·la comprova si la millora no depèn només d'haver ajustat i avaluat sobre les mateixes dades. En cada fold, els coeficients de \(T_{rad}\) i \(T_{env}^{*}\) s'ajusten amb unes caminades i s'avaluen en caminades no vistes.

In [ ]:
# ============================================================
# 7. Validació creuada per caminada
# ============================================================
def cv_by_walk(v, outcome, n_splits=5):
    v, target_name, direction = add_binary_target(v, outcome)
    groups = v['walk'].values
    gkf = GroupKFold(n_splits=min(n_splits, pd.Series(groups).nunique()))

    rows = []
    for fold, (tr, te) in enumerate(gkf.split(v, v['score'], groups), start=1):
        train = v.iloc[tr].copy()
        test = v.iloc[te].copy()

        rad = fit_T_rad(train, direction, K_GRID_CV)
        env = fit_T_env(train, direction, K_GRID_CV)
        test2 = apply_models(test, rad, env)

        for var in ['T_corr', 'HDX_corr', 'T_rad', 'T_env']:
            raw = spearman(test2[var], test2['score'])
            rows.append({
                'outcome': outcome,
                'fold': fold,
                'variable': var,
                'rho_raw_test': raw,
                'objective_rho_test': direction * raw,
                'AUC_binary_test': auc_from_coordinate(test2[var], test2['binary_target']),
                'Kruskal_H_state3_test': kw_state3(test2[var], test2['state3']),
                'k_mix_rad': rad['k_mix'],
                'k_sun_rad': rad['k_sun'],
                'k_mix_env': env['k_mix'],
                'k_sun_env': env['k_sun'],
                'k_wind_env': env['k_wind'],
            })

    cv = pd.DataFrame(rows)
    return cv

if RUN_CV:
    cv_sens = cv_by_walk(v_sens, 'sensation')
    cv_comf = cv_by_walk(v_comf, 'comfort')
    cv_all = pd.concat([cv_sens, cv_comf], ignore_index=True)
    cv_all.to_csv(OUTDIR / 'cv_by_walk_results.csv', index=False)

    cv_summary = (
        cv_all.groupby(['outcome', 'variable'])
        .agg(
            objective_rho_mean=('objective_rho_test', 'mean'),
            objective_rho_std=('objective_rho_test', 'std'),
            AUC_mean=('AUC_binary_test', 'mean'),
            AUC_std=('AUC_binary_test', 'std'),
            KW_H_mean=('Kruskal_H_state3_test', 'mean'),
        )
        .reset_index()
    )
    cv_summary.to_csv(OUTDIR / 'cv_by_walk_summary.csv', index=False)
    display(cv_summary)
else:
    print('RUN_CV = False')

## Regressió ordinal: només com a comprovació de signes

La regressió ordinal no es fa servir com a definició final de \(T_{env}^{*}\). Serveix per comprovar si els signes són coherents:

- Sensació: temperatura i sol haurien de desplaçar cap a vots més calents; vent cap a vots més freds.
- Comfort: temperatura i sol haurien de reduir comfort; vent hauria d'augmentar-lo en condicions càlides.

In [ ]:
# ============================================================
# 8. Regressió ordinal: validació dels signes
# ============================================================
def ordinal_sign_check(v, outcome):
    try:
        from statsmodels.miscmodels.ordinal_model import OrderedModel
    except Exception as e:
        print('statsmodels OrderedModel no disponible:', e)
        return None

    vv = v.dropna(subset=['score', 'T_corr', 'f_mix', 'f_sun', 'wind_sc']).copy()
    y = pd.Categorical(vv['score'], categories=sorted(vv['score'].unique()), ordered=True)
    X = vv[['T_corr', 'f_mix', 'f_sun', 'wind_sc']]

    model = OrderedModel(y, X, distr='logit')
    res = model.fit(method='bfgs', disp=False, maxiter=500)

    pred_cols = ['T_corr', 'f_mix', 'f_sun', 'wind_sc']
    coefs = res.params.loc[pred_cols].to_frame('coef')
    coefs['sign'] = np.sign(coefs['coef']).map({-1.0: '-', 0.0: '0', 1.0: '+'})
    coefs['outcome'] = outcome

    beta_T = coefs.loc['T_corr', 'coef']
    if np.isfinite(beta_T) and abs(beta_T) > 1e-12:
        # Equivalents en graus dins de la mateixa escala lineal del model ordinal.
        coefs.loc['f_mix', 'degree_equiv'] = coefs.loc['f_mix', 'coef'] / beta_T
        coefs.loc['f_sun', 'degree_equiv'] = coefs.loc['f_sun', 'coef'] / beta_T
        coefs.loc['wind_sc', 'degree_equiv'] = -coefs.loc['wind_sc', 'coef'] / beta_T
    else:
        coefs['degree_equiv'] = np.nan

    return coefs.reset_index(names='term')

ord_sens = ordinal_sign_check(v_sens, 'sensation')
ord_comf = ordinal_sign_check(v_comf, 'comfort')

ordinal_results = pd.concat([x for x in [ord_sens, ord_comf] if x is not None], ignore_index=True)
ordinal_results.to_csv(OUTDIR / 'ordinal_sign_check.csv', index=False)
display(ordinal_results)

# Missatge final per defensar

1. \(T_{corr}\) és la variable física base.
2. Afegir exposició solar produeix \(T_{rad}\) i millora clarament la separació dels vots.
3. Afegir vent produeix \(T_{env}^{*}\), que és la millor coordenada empírica.
4. \(T_{env}^{*}\) explica més directament la **sensació tèrmica** que el comfort, perquè la sensació és una resposta tèrmica ordinal directa.
5. El comfort també millora, però s'interpreta com a resposta secundària perquè inclou preferència, adaptació i context.
6. Les \(k\) són pesos empírics dins d'un grid finit, no graus físics exactes.
7. La regressió ordinal confirma els signes físics esperats: sol escalfa, vent refreda; en comfort, sol i calor redueixen comfort i el vent l'augmenta.

In [ ]:
# ============================================================
# 9. Guardar taula final compacta
# ============================================================
coef_all = pd.concat([coef_sens, coef_comf], ignore_index=True)
coef_all.to_csv(OUTDIR / 'grid_coefficients_all.csv', index=False)

print('Resultats guardats a:', OUTDIR.resolve())
print('\nFitxers principals:')
for p in sorted(OUTDIR.glob('*')):
    print(' -', p.name)